In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import kagglehub
print(f'PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')

In [ ]:
import os
import glob
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

def augment(mixture, sources):
    gain = random.uniform(0.7, 1.0)
    flip = random.random() < 0.5
    s = -1.0 if flip else 1.0
    mixture = mixture * (s * gain)
    sources = sources * (s * gain)
    if random.random() < 0.3:
        noise = torch.randn_like(mixture) * 0.005
        mixture = mixture + noise
    return (mixture, sources)

class MultiSpeakerMixtureDataset(Dataset):

    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.indices = sorted((int(os.path.basename(p).split('_')[1].split('.')[0]) for p in glob.glob(os.path.join(data_dir, 'mix_*.npy'))))
        if len(self.indices) == 0:
            raise ValueError(f'No mix_*.npy files found in {data_dir}.')
        manifest = np.load(os.path.join(data_dir, 'manifest.npy'), allow_pickle=True)
        self.n_speakers = {int(m['idx']): int(m['n_speakers']) for m in manifest}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        mix_np = np.load(os.path.join(self.data_dir, f'mix_{idx:06d}.npy'), mmap_mode='r')
        src_np = np.load(os.path.join(self.data_dir, f'sources_{idx:06d}.npy'), mmap_mode='r')
        mixture = torch.from_numpy(np.array(mix_np, copy=True)).float()
        sources = torch.from_numpy(np.array(src_np, copy=True)).float()
        (mixture, sources) = augment(mixture, sources)
        return {'mixture': mixture, 'sources': sources, 'n_speakers': sources.shape[0]}

def variable_speaker_collate(batch):
    mixtures = torch.stack([b['mixture'] for b in batch], dim=0)
    sources_list = [b['sources'] for b in batch]
    n_speakers = [b['n_speakers'] for b in batch]
    return {'mixture': mixtures, 'sources_list': sources_list, 'n_speakers': n_speakers}

def make_speaker_sorted_loader(dataset, batch_size, shuffle=True, **kwargs):
    if isinstance(dataset, torch.utils.data.Subset):
        base_dataset = dataset.dataset
        subset_positions = dataset.indices
    else:
        base_dataset = dataset
        subset_positions = list(range(len(dataset)))
    groups = defaultdict(list)
    for (pos_in_subset, pos_in_base) in enumerate(subset_positions):
        file_idx = base_dataset.indices[pos_in_base]
        n = base_dataset.n_speakers.get(file_idx, 2)
        groups[n].append(pos_in_subset)
    batches = []
    for n in sorted(groups.keys()):
        idxs = groups[n]
        if shuffle:
            random.shuffle(idxs)
        for start in range(0, len(idxs), batch_size):
            batch = idxs[start:start + batch_size]
            if len(batch) > 0:
                batches.append(batch)
    if shuffle:
        random.shuffle(batches)
    return DataLoader(dataset, batch_sampler=batches, collate_fn=variable_speaker_collate, **kwargs)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ResidualConvBlock(nn.Module):

    def __init__(self, channels, kernel_size=3, dilation=1):
        super().__init__()
        pad = dilation * (kernel_size - 1) // 2
        self.net = nn.Sequential(nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation, groups=channels), nn.Conv1d(channels, channels, 1), nn.GroupNorm(8, channels), nn.PReLU())

    def forward(self, x):
        return x + self.net(x)

class CrossAttentionAttractor(nn.Module):

    def __init__(self, dim, n_heads=4, pool_len=64):
        super().__init__()
        self.pool_len = pool_len
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attn_resid = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.attn_mix = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.history_gru = nn.GRUCell(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.fuse = nn.Linear(dim * 3, dim)

    def _pool(self, x):
        return F.adaptive_avg_pool1d(x, self.pool_len).permute(0, 2, 1)

    def forward(self, residual, mixture_latent, history_state):
        r = self._pool(residual)
        m = self._pool(mixture_latent)
        q = self.query.expand(r.size(0), -1, -1)
        (from_resid, _) = self.attn_resid(q, r, r)
        (from_mix, _) = self.attn_mix(q, m, m)
        combined = torch.cat([from_resid.squeeze(1), from_mix.squeeze(1), history_state], dim=-1)
        fingerprint = self.fuse(combined)
        new_history = self.history_gru(fingerprint, history_state)
        return (self.norm(fingerprint), new_history)

class AttractorGuidedSeparator(nn.Module):

    def __init__(self, encoder_dim=512, bottleneck_dim=256, n_heads=8, n_blocks=6):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.bottleneck_dim = bottleneck_dim
        self.encoder_front = nn.Sequential(nn.Conv1d(1, encoder_dim, kernel_size=16, stride=8, padding=4), nn.GroupNorm(8, encoder_dim), nn.PReLU())
        self.encoder_deep = nn.Sequential(ResidualConvBlock(encoder_dim, dilation=1), ResidualConvBlock(encoder_dim, dilation=2), ResidualConvBlock(encoder_dim, dilation=4), ResidualConvBlock(encoder_dim, dilation=8))
        self.attractor = CrossAttentionAttractor(encoder_dim, n_heads=4, pool_len=64)
        self.film_gamma = nn.Linear(encoder_dim, encoder_dim)
        self.film_beta = nn.Linear(encoder_dim, encoder_dim)
        self.bottleneck_in = nn.Conv1d(encoder_dim, bottleneck_dim, 1)
        self.bottleneck_out = nn.Conv1d(bottleneck_dim, encoder_dim, 1)
        self.separator_blocks = nn.Sequential(*[nn.TransformerEncoderLayer(d_model=bottleneck_dim, nhead=n_heads, dim_feedforward=1024, dropout=0.1, batch_first=True, norm_first=True) for _ in range(n_blocks)])
        self.dilated_recurrent = nn.Sequential(nn.Conv1d(bottleneck_dim, bottleneck_dim, 3, padding=1, dilation=1, groups=bottleneck_dim), nn.Conv1d(bottleneck_dim, bottleneck_dim, 1), nn.PReLU(), nn.Conv1d(bottleneck_dim, bottleneck_dim, 3, padding=2, dilation=2, groups=bottleneck_dim), nn.Conv1d(bottleneck_dim, bottleneck_dim, 1), nn.PReLU(), nn.Conv1d(bottleneck_dim, bottleneck_dim, 3, padding=4, dilation=4, groups=bottleneck_dim), nn.Conv1d(bottleneck_dim, bottleneck_dim, 1), nn.PReLU())
        self.mask_gen = nn.Sequential(nn.Conv1d(encoder_dim, encoder_dim, 1), nn.PReLU(), nn.Conv1d(encoder_dim, encoder_dim, 1), nn.Sigmoid())
        self.decoder_proj = nn.Conv1d(encoder_dim * 2, encoder_dim, 1)
        self.decoder = nn.ConvTranspose1d(encoder_dim, 1, kernel_size=16, stride=8, padding=4)
        self.stopper = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(encoder_dim, 64), nn.PReLU(), nn.Linear(64, 1))
        self.register_buffer('pe', self._build_pe(10000, bottleneck_dim), persistent=False)

    def _build_pe(self, max_len, dim):
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def encode(self, waveform):
        return self.encoder_deep(self.encoder_front(waveform))

    def extract_one(self, residual_latent, mixture_latent, original_length=None, history_state=None):
        if history_state is None:
            history_state = torch.zeros(residual_latent.size(0), self.encoder_dim, device=residual_latent.device)
        (voice_fingerprint, new_history) = self.attractor(residual_latent, mixture_latent, history_state)
        gamma = self.film_gamma(voice_fingerprint).unsqueeze(-1)
        beta = self.film_beta(voice_fingerprint).unsqueeze(-1)
        conditioned = residual_latent * gamma + beta
        x = self.bottleneck_in(conditioned).permute(0, 2, 1)
        T = x.size(1)
        x = x + self.pe[:, :T, :].to(x.device)
        x = self.separator_blocks(x)
        x = x.permute(0, 2, 1)
        x = self.dilated_recurrent(x) + x
        sep_latent = self.bottleneck_out(x)
        mask = self.mask_gen(sep_latent)
        spk_latent = residual_latent * mask
        next_residual_latent = residual_latent - spk_latent
        decode_input = torch.cat([spk_latent, mixture_latent], dim=1)
        decode_input = self.decoder_proj(decode_input)
        dominant_audio = self.decoder(decode_input).squeeze(1)
        if original_length is not None:
            L = dominant_audio.shape[-1]
            if L > original_length:
                dominant_audio = dominant_audio[..., :original_length]
            elif L < original_length:
                dominant_audio = F.pad(dominant_audio, (0, original_length - L))
        stop_logit = self.stopper(spk_latent)
        return (dominant_audio, stop_logit, next_residual_latent, new_history)

    def forward(self, waveform):
        if waveform.dim() == 2:
            waveform = waveform.unsqueeze(1)
        original_length = waveform.shape[-1]
        mixture_latent = self.encode(waveform)
        residual_latent = mixture_latent.clone()
        history_state = torch.zeros(waveform.size(0), self.encoder_dim, device=waveform.device)
        separated = []
        for _ in range(6):
            (audio, stop_logit, residual_latent, history_state) = self.extract_one(residual_latent, mixture_latent, original_length, history_state)
            separated.append(audio)
            if torch.sigmoid(stop_logit).mean().item() < 0.5:
                break
        return separated

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiscaleSTFTLoss(nn.Module):

    def __init__(self, fft_sizes=(256, 512, 1024)):
        super().__init__()
        self.fft_sizes = fft_sizes
        for n_fft in fft_sizes:
            self.register_buffer(f'win_{n_fft}', torch.hann_window(n_fft), persistent=False)

    def forward(self, estimate, target):
        loss = torch.zeros(estimate.size(0), device=estimate.device)
        for n_fft in self.fft_sizes:
            hop = n_fft // 4
            win = getattr(self, f'win_{n_fft}')
            est_mag = torch.stft(estimate, n_fft, hop, window=win, return_complex=True).abs()
            tgt_mag = torch.stft(target, n_fft, hop, window=win, return_complex=True).abs()
            diff_norm = torch.sqrt(((tgt_mag - est_mag) ** 2).sum(dim=[-2, -1]) + 1e-08)
            tgt_norm = torch.sqrt((tgt_mag ** 2).sum(dim=[-2, -1]) + 1e-08)
            l1_loss = F.l1_loss(torch.log(est_mag + 1e-07), torch.log(tgt_mag + 1e-07), reduction='none').mean(dim=[-2, -1])
            loss = loss + diff_norm / tgt_norm + l1_loss
        return loss / len(self.fft_sizes)

def train_batch_variable_speakers(model, optimizer, mixtures, sources_list, stft_loss_fn, max_iterations=6, stop_weight=0.1, epoch=0):
    device = mixtures.device
    model.train()
    optimizer.zero_grad(set_to_none=True)
    B = mixtures.size(0)
    K = sources_list[0].size(0)
    original_length = mixtures.size(-1)
    sources = torch.stack(sources_list).to(device, non_blocking=True)
    mixture_latent = model.encode(mixtures.unsqueeze(1))
    with torch.no_grad():
        src_latents = model.encode(sources.view(B * K, 1, -1)).view(B, K, model.encoder_dim, -1)
    residual_latent = mixture_latent.clone()
    history = torch.zeros(B, model.encoder_dim, device=device)
    alive = torch.ones(B, K, dtype=torch.bool, device=device)
    total_loss = torch.zeros(1, device=device)
    for step in range(max_iterations):
        active_mask = alive.sum(dim=-1) > 0
        (audio, stop_logit, next_residual, history) = model.extract_one(residual_latent, mixture_latent, original_length, history)
        est = audio.unsqueeze(1).expand(-1, K, -1)
        est_zm = est - est.mean(dim=-1, keepdim=True)
        tgt_zm = sources - sources.mean(dim=-1, keepdim=True)
        dot = (est_zm * tgt_zm).sum(dim=-1)
        energy = (tgt_zm ** 2).sum(dim=-1) + 1e-08
        proj = dot.unsqueeze(-1) / energy.unsqueeze(-1) * tgt_zm
        noise = est_zm - proj
        ratio = proj.pow(2).sum(dim=-1) / (noise.pow(2).sum(dim=-1) + 1e-08)
        sisdr = -10 * torch.log10(ratio + 1e-08)
        sisdr_masked = torch.where(alive, sisdr, torch.tensor(float('inf'), device=device))
        best_k = sisdr_masked.argmin(dim=-1)
        best_sisdr = sisdr_masked[torch.arange(B, device=device), best_k]
        best_target = sources[torch.arange(B, device=device), best_k, :]
        stft_val = stft_loss_fn(audio.float(), best_target.float())
        step_audio_loss = torch.where(active_mask, best_sisdr + 0.5 * stft_val, torch.zeros_like(best_sisdr))
        speakers_left_after = alive.sum(dim=-1) - 1
        stop_label = (speakers_left_after > 0).float().unsqueeze(-1)
        bce_loss = F.binary_cross_entropy_with_logits(stop_logit, stop_label, reduction='none').squeeze(-1)
        step_total = step_audio_loss + stop_weight * bce_loss
        total_loss = total_loss + step_total.mean()
        step_extracted = F.one_hot(best_k, num_classes=K).bool()
        alive = alive & ~step_extracted
        batch_idx = torch.arange(B, device=device)
        tf_prob = max(0.0, 1.0 - epoch / 10)
        tf_mask = (torch.rand(B, device=device) < tf_prob) & active_mask
        gt_residual = mixture_latent - src_latents[batch_idx, best_k, :, :]
        next_latents = torch.where(tf_mask.view(B, 1, 1), gt_residual, next_residual.detach())
        residual_latent = next_latents
    final_loss = total_loss / max_iterations
    final_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()
    return final_loss.item()

In [ ]:
import torch.optim as optim
import time
import os

def run_training():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on: {(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')}")
    data_dir = '/kaggle/input/datasets/dakshgoyal387/librispeech-mixtures'
    full_dataset = MultiSpeakerMixtureDataset(data_dir)
    val_size = int(0.1 * len(full_dataset))
    train_size = len(full_dataset) - val_size
    (train_dataset, val_dataset) = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
    model = AttractorGuidedSeparator(encoder_dim=512, bottleneck_dim=256, n_heads=8, n_blocks=6).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    stft_loss_fn = MultiscaleSTFTLoss(fft_sizes=(256, 512, 1024)).to(device)
    warmup_epochs = 2
    warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    plateau_sched = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    epochs = 30
    max_iterations = 6
    best_val_loss = float('inf')
    start_epoch = 0
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    ckpt_path = '/kaggle/working/checkpoints/best_model.pt'
    if os.path.exists(ckpt_path):
        print(f'Found checkpoint at {ckpt_path}. Resuming training...')
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scaler_state_dict' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt['val_loss']
        print(f'Resuming from Epoch {start_epoch} | Previous best val loss: {best_val_loss:.4f}')
    else:
        print('No checkpoint found. Starting from scratch.')
    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        train_loader = make_speaker_sorted_loader(train_dataset, batch_size=12, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
        val_loader = make_speaker_sorted_loader(val_dataset, batch_size=12, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
        train_loss = 0.0
        model.train()
        for (batch_idx, batch) in enumerate(train_loader):
            mixtures = batch['mixture'].to(device, non_blocking=True)
            sources_list = batch['sources_list']
            loss = train_batch_variable_speakers(model, optimizer, mixtures, sources_list, stft_loss_fn, max_iterations, epoch=epoch)
            train_loss += loss
            if batch_idx % 20 == 0:
                print(f'  [Epoch {epoch + 1} | Batch {batch_idx}/{len(train_loader)}] loss: {loss:.4f}')
        avg_train = train_loss / len(train_loader)
        model.eval()
        val_loss_accum = 0.0
        n_val_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                mixtures = batch['mixture'].to(device, non_blocking=True)
                sources = torch.stack(batch['sources_list']).to(device, non_blocking=True)
                original_length = mixtures.shape[-1]
                mixture_latent = model.encode(mixtures.unsqueeze(1))
                B = mixtures.size(0)
                K = sources.size(1)
                latent = mixture_latent.clone()
                history_state = torch.zeros(B, model.encoder_dim, device=device)
                alive = torch.ones(B, K, dtype=torch.bool, device=device)
                batch_ex_loss = torch.zeros(B, device=device)
                active_steps = torch.zeros(B, device=device)
                for step in range(max_iterations):
                    active_mask = alive.sum(dim=-1) > 0
                    if not active_mask.any():
                        break
                    (audio, _, next_latent, history_state) = model.extract_one(latent, mixture_latent, original_length, history_state)
                    est = audio.unsqueeze(1).expand(-1, K, -1)
                    est_zm = est - est.mean(dim=-1, keepdim=True)
                    tgt_zm = sources - sources.mean(dim=-1, keepdim=True)
                    dot = (est_zm * tgt_zm).sum(dim=-1)
                    energy = (tgt_zm ** 2).sum(dim=-1) + 1e-08
                    proj = dot.unsqueeze(-1) / energy.unsqueeze(-1) * tgt_zm
                    noise = est_zm - proj
                    ratio = proj.pow(2).sum(dim=-1) / (noise.pow(2).sum(dim=-1) + 1e-08)
                    sisdr = -10 * torch.log10(ratio + 1e-08)
                    sisdr_masked = torch.where(alive, sisdr, torch.tensor(float('inf'), device=device))
                    best_k = sisdr_masked.argmin(dim=-1)
                    best_sisdr = sisdr_masked[torch.arange(B, device=device), best_k]
                    batch_ex_loss = batch_ex_loss + torch.where(active_mask, best_sisdr, torch.zeros_like(best_sisdr))
                    active_steps = active_steps + active_mask.float()
                    step_extracted = F.one_hot(best_k, num_classes=K).bool()
                    alive = alive & ~step_extracted
                    latent = next_latent
                val_loss_accum += (batch_ex_loss / torch.clamp(active_steps, min=1.0)).mean().item()
                n_val_batches += 1
        avg_val = val_loss_accum / max(n_val_batches, 1)
        print(f'Epoch {epoch + 1}/{epochs} | train: {avg_train:.4f} | val: {avg_val:.4f} | {time.time() - t0:.1f}s')
        if epoch < warmup_epochs:
            warmup_sched.step()
        else:
            plateau_sched.step(avg_val)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            if os.path.exists(ckpt_path):
                os.remove(ckpt_path)
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_loss': avg_val}, ckpt_path)
            print(f'  ** New best: {avg_val:.4f}')
run_training()